# Tome C2, Chapter 2: Sorting, Selection and Sorting Networks (Python companion)

Exercises first, solutions at the end. We check merge sort's exact worst case against brute force, draw the distribution of quicksort's comparisons, race three selection algorithms, sort by digits, build Batcher's networks and verify them on every $0$–$1$ input, and run the horse race of the opening puzzle.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math, time
import numpy as np
import matplotlib.pyplot as plt
random.seed(2); rng = np.random.default_rng(2)
lg = math.log2

## Exercises

**P2.1 (★) Merge sort, exactly.** Count merge sort's comparisons on every ordering of $n\le 8$ elements, and check that the maximum is $W(n)=\sum_{k\le n}\lceil\log_2k\rceil$ (Theorem 2.2). Plot $W(n)-\log_2 n!$.

**P2.2 (★) Quicksort's distribution.** Run randomised quicksort many times on $n=1000$ and $n=20000$ elements. Compare the mean with $2(n+1)H_n-4n$, draw the histogram of $(C_n-\mathbb E C_n)/n$, and compare the deepest recursion seen with the bound $\lceil 30\ln n\rceil+1$ of Theorem 2.4.

**P2.3 (★★) Selection.** Implement quickselect, the median of medians (Theorem 2.6) and sampling selection (Theorem 2.7), all counting comparisons. Compare their counts for the median with $8n$, $2n$ and $1.5n$, check Exercise 2.2 ($2n-2H_n$ for the minimum), and measure the failure rate of a sampling trial.

**P2.4 (★) Radix sort.** Implement counting sort and LSD radix sort, sort a million integers below $10^{12}$ with base $2^{16}$, and check the role of stability by breaking it.

**P2.5 (★★) Sorting networks.** Generate Batcher's bitonic and odd–even merge networks, count their comparators and depth, and verify them on all $2^n$ inputs of $0$s and $1$s for $n\le 16$ (Theorem 2.11). Check Exercises 2.6–2.9, and find the smallest $0$–$1$ counterexample for a network with one comparator removed.

**P2.6 (★) Horses.** Simulate the seven-race method of Exercise 2.10 on random speeds and check that it always finds the three fastest.

## Solutions

### P2.1 Merge sort, exactly

In [ ]:
def merge_sort(xs, cnt):
    if len(xs) <= 1: return xs
    m = (len(xs) + 1) // 2                      # left half has ceil(n/2) elements, as in the text
    A, B = merge_sort(xs[:m], cnt), merge_sort(xs[m:], cnt)
    out, i, j = [], 0, 0
    while i < len(A) and j < len(B):
        cnt[0] += 1
        if A[i] < B[j]: out.append(A[i]); i += 1
        else: out.append(B[j]); j += 1
    return out + A[i:] + B[j:]

W = lambda n: sum(math.ceil(lg(k)) for k in range(1, n + 1))
for n in range(1, 9):
    worst = 0
    for p in itertools.permutations(range(n)):
        c = [0]; merge_sort(list(p), c); worst = max(worst, c[0])
    t = math.ceil(lg(n)) if n > 1 else 0
    assert worst == W(n) == n * t - 2 ** t + 1
    print(f"n={n}: worst = {worst} = W(n);  ceil(log2 n!) = {math.ceil(lg(math.factorial(n)))}")
ns = np.arange(1, 2049)
gap = [W(n) - lg(math.factorial(n)) for n in ns]
plt.figure(figsize=(8, 3)); plt.plot(ns, np.array(gap) / ns, lw=0.8)
plt.xlabel("n"); plt.ylabel(r"$(W(n)-\log_2 n!)/n$"); plt.title("merge sort's loss per element"); plt.show()

The loss per element oscillates between powers of $2$, with values between about $0.92$ and $1.44$ for large $n$ (since $\log_2 n!=n\log_2n-n\log_2e+O(\log n)$ while $W(n)=n\lceil\log_2n\rceil-2^{\lceil\log_2n\rceil}+1$).

### P2.2 Quicksort's distribution

An iterative implementation that records, for each run, the number of comparisons and the depth of the recursion.

In [ ]:
def quicksort_stats(a):
    comps, maxdepth = 0, 0
    stack = [(a, 1)]
    while stack:
        a, d = stack.pop()
        if len(a) <= 1:
            continue
        maxdepth = max(maxdepth, d)
        p = a[random.randrange(len(a))]
        comps += len(a) - 1
        stack.append(([x for x in a if x < p], d + 1)); stack.append(([x for x in a if x > p], d + 1))
    return comps, maxdepth

H = lambda n: sum(1 / k for k in range(1, n + 1))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for idx, (n, runs) in enumerate([(1000, 3000), (20000, 300)]):
    data = [quicksort_stats(list(range(n))) for _ in range(runs)]
    C = np.array([c for c, d in data]); D = [d for c, d in data]
    mean = 2 * (n + 1) * H(n) - 4 * n
    print(f"n={n}: mean {C.mean():.0f} (theory {mean:.0f}), sd/n = {C.std()/n:.3f}, max/mean = {C.max()/mean:.3f}, "
          f"deepest recursion {max(D)} vs ln n = {math.log(n):.1f} and the bound {math.ceil(30*math.log(n))+1}")
    ax[idx].hist((C - mean) / n, bins=40, color="C0"); ax[idx].set_title(f"$(C_n-EC_n)/n$, n={n}")
plt.tight_layout(); plt.show()

The standard deviation is a constant times $n$ ($\sqrt{7-2\pi^2/3}\,n\approx0.65n$, a classical result), so relative to the mean $\approx 2n\ln n$ the spread shrinks. The histograms have the same skewed shape for both sizes: that is Rösler's limit law. The deepest recursion is a small multiple of $\ln n$ (about $4.3\ln n$ in the limit, by a theorem of Devroye), far below the lavish bound of Theorem 2.4.

### P2.3 Selection

All three functions take a list of distinct numbers and $k$ (1-based) and count comparisons in a shared counter. In the median of medians a group of five is sorted with insertion sort (at most ten comparisons; Exercise 1.8 would do with seven).

In [ ]:
class Counter:
    def __init__(self): self.c = 0
    def less(self, a, b): self.c += 1; return a < b

def quickselect(a, k, C):
    while True:
        if len(a) == 1: return a[0]
        p = a[random.randrange(len(a))]
        lo, hi = [], []
        for x in a:
            if x == p: continue
            (lo if C.less(x, p) else hi).append(x)
        if k <= len(lo): a = lo
        elif k == len(lo) + 1: return p
        else: k -= len(lo) + 1; a = hi

def isort(g, C):
    g = list(g)
    for i in range(1, len(g)):
        j = i
        while j > 0 and C.less(g[j], g[j - 1]): g[j], g[j - 1] = g[j - 1], g[j]; j -= 1
    return g

def mom_select(a, k, C):
    if len(a) < 140:
        return isort(a, C)[k - 1]
    meds = [isort(a[i:i + 5], C)[(len(a[i:i + 5]) - 1) // 2] for i in range(0, len(a), 5)]
    m = mom_select(meds, (len(meds) + 1) // 2, C)
    lo, hi = [], []
    for x in a:
        if x == m: continue
        (lo if C.less(x, m) else hi).append(x)
    if k <= len(lo): return mom_select(lo, k, C)
    if k == len(lo) + 1: return m
    return mom_select(hi, k - len(lo) - 1, C)

def sample_select(a, k, C, stats=None):
    n = len(a)
    while True:
        r = math.ceil(n ** 0.75)
        R = sorted(random.choice(a) for _ in range(r)); C.c += int(r * max(1, lg(r)))   # charge the sample sort
        x = k * r / n; l = math.floor(x - n ** 0.5); h = math.ceil(x + n ** 0.5)
        lo_b = R[l - 1] if l >= 1 else -math.inf; hi_b = R[h - 1] if h <= r else math.inf
        t, P = 0, []
        for y in a:
            if k <= n / 2:
                if C.less(hi_b, y): continue
                if C.less(y, lo_b): t += 1
                else: P.append(y)
            else:
                if C.less(y, lo_b): t += 1; continue
                if not C.less(hi_b, y): P.append(y)
        if t < k <= t + len(P) and len(P) <= 4 * n ** 0.75 + 2:
            P.sort(); C.c += int(len(P) * max(1, lg(max(2, len(P)))))
            return P[k - t - 1]
        if stats is not None: stats["fail"] += 1

for n in (10001, 100001, 1000001):
    a = random.sample(range(10 ** 8), n); med = sorted(a)[n // 2]
    row = []
    for name, f, reps in [("quickselect", quickselect, 5), ("median of medians", mom_select, 1), ("sampling", sample_select, 5)]:
        counts = []
        for _ in range(reps):
            C = Counter(); assert f(list(a), n // 2 + 1, C) == med; counts.append(C.c)
        row.append(f"{name} {np.mean(counts)/n:5.2f}n")
    print(f"median of n={n:8d}:  " + "   ".join(row))

Quickselect lands near the exact value $(2+2\ln2)n\approx3.39n$; the median of medians pays dearly for its guarantee. Sampling starts badly, because the sample and the set $P$, of sizes about $n^{3/4}$ and $2n^{3/4}$, are a large fraction of small inputs and must be sorted; as $n$ grows its count falls towards $1.5n$, below the $2n$ that Bent and John proved necessary for deterministic algorithms, but slowly: the $o(n)$ of Theorem 2.7 is $O(n^{3/4}\log n)$, and $n^{-1/4}\log_2 n$ is still about $0.6$ at $n=10^6$. Floyd and Rivest's version uses a sample of size about $n^{2/3}$ and a narrower band, and reaches $1.5n+o(n)$ much sooner.

Exercise 2.2, and the failure rate of one sampling trial:

In [ ]:
for n in (10, 100, 1000):
    tot = 0
    for _ in range(4000):
        C = Counter(); quickselect(list(range(n)), 1, C); tot += C.c
    print(f"n={n:5d}: mean comparisons for the minimum {tot/4000:9.2f}   2n-2H_n = {2*n-2*H(n):9.2f}")
for n in (1000, 10000, 100000):
    st = {"fail": 0}; a = list(range(n)); trials = 0
    for _ in range(100):
        C = Counter(); sample_select(a, n // 2, C, st); trials += 1
    print(f"n={n:6d}: failed trials per success {st['fail']/trials:.3f}   (bound 8 n^(-1/4) = {8*n**-0.25:.2f})")

### P2.4 Radix sort

In [ ]:
def counting_sort(keys, digit_of, B):
    cnt = [0] * B
    for x in keys: cnt[digit_of(x)] += 1
    pos, s = [0] * B, 0
    for j in range(B): pos[j] = s; s += cnt[j]
    out = [None] * len(keys)
    for x in keys:
        d = digit_of(x); out[pos[d]] = x; pos[d] += 1
    return out

def radix_sort(keys, B=1 << 16, d=3):
    for i in range(d):
        keys = counting_sort(keys, lambda x, i=i: (x // B ** i) % B, B)
    return keys

xs = [random.randrange(10 ** 12) for _ in range(10 ** 6)]
t0 = time.perf_counter(); ys = radix_sort(xs); t1 = time.perf_counter(); zs = sorted(xs); t2 = time.perf_counter()
assert ys == zs
print(f"radix sort {t1-t0:.2f} s in pure Python (3 passes), built-in sorted {t2-t1:.2f} s in C")

def unstable_pass(keys, digit_of, B):          # same buckets, but each bucket reversed: not stable
    buckets = [[] for _ in range(B)]
    for x in keys: buckets[digit_of(x)].append(x)
    return [x for b in buckets for x in reversed(b)]
small = [random.randrange(100) for _ in range(20)]
k = small
for i in range(2): k = unstable_pass(k, lambda x, i=i: (x // 10 ** i) % 10, 10)
print("with an unstable pass:", k[:10], "... sorted?", k == sorted(small))

In pure Python the interpreter's overhead hides the asymptotic advantage; written in C, radix sort beats comparison sorting for large arrays of machine integers. Without stability the second pass destroys the work of the first.

### P2.5 Sorting networks

A network is a list of rounds, each a list of comparators $(i,j)$ with $i<j$ (0-based). We check all $0$–$1$ inputs at once with numpy: an array of shape $2^n\times n$.

In [ ]:
def bitonic(n):
    layers, k = [], 2
    while k <= n:
        layers.append([(b + i, b + k - 1 - i) for b in range(0, n, k) for i in range(k // 2)])
        j = k // 4
        while j >= 1:
            layers.append([(b + i, b + i + j) for b in range(0, n, 2 * j) for i in range(j)])
            j //= 2
        k *= 2
    return layers

def oddeven(n):
    # Batcher's odd-even merge sort, as a list of comparators in a valid order
    comps = []
    def merge(lo, m, r):                       # merge the two sorted halves of wires lo, lo+r, lo+2r, ... (m wires)
        step = 2 * r
        if step < m:
            merge(lo, m, step); merge(lo + r, m, step)
            comps.extend((i, i + r) for i in range(lo + r, lo + m - r, step))
        else:
            comps.append((lo, lo + r))
    def sort(lo, m):
        if m > 1:
            sort(lo, m // 2); sort(lo + m // 2, m // 2); merge(lo, m, 1)
    sort(0, n)
    return [[c] for c in comps]

def apply(net, X):
    X = X.copy()
    for L in net:
        for i, j in L:
            a, b = X[:, i].copy(), X[:, j]
            X[:, i] = np.minimum(a, b); X[:, j] = np.maximum(a, b)
    return X

def all01(n):
    return ((np.arange(1 << n)[:, None] >> np.arange(n)) & 1).astype(np.uint8)

def sorts01(net, n):
    Y = apply(net, all01(n))
    return bool(np.all(Y[:, :-1] <= Y[:, 1:]))

def depth(net, n):
    t = [0] * n; d = 0
    for L in net:
        for i, j in L:
            t[i] = t[j] = max(t[i], t[j]) + 1; d = max(d, t[i])
    return d

for n in (2, 4, 8, 16):
    p = int(lg(n))
    b, o = bitonic(n), oddeven(n)
    nb_, no_ = sum(map(len, b)), sum(map(len, o))
    assert sorts01(b, n) and sorts01(o, n)
    assert nb_ == n * p * (p + 1) // 4 and no_ == n * p * (p - 1) // 4 + n - 1
    print(f"n={n:2d}: bitonic {nb_:3d} comparators, depth {depth(b, n):2d};  odd-even {no_:3d} comparators, depth {depth(o, n):2d};  "
          f"both sort all {1<<n} 0-1 inputs")

The zero–one principle in action: $65536$ binary inputs certify a $16$-wire network that would otherwise need $16!\approx2\times10^{13}$ permutations. Now break a network and look for the smallest failing input, and check Exercises 2.6–2.8.

In [ ]:
n = 8; net = bitonic(n)
flat = [c for L in net for c in L]
for drop in (0, 9, 20):
    broken = [[c] for idx, c in enumerate(flat) if idx != drop]
    Y = apply(broken, all01(n)); bad = np.where(np.any(Y[:, :-1] > Y[:, 1:], axis=1))[0]
    x = all01(n)[bad[0]]
    print(f"remove comparator {flat[drop]}: {len(bad)} failing 0-1 inputs, e.g. {x.tolist()} -> {apply(broken, x[None])[0].tolist()}")

four = [[(0, 1), (2, 3)], [(0, 2), (1, 3)], [(1, 2)]]
assert sorts01(four, 4) and depth(four, 4) == 3
print("Exercise 2.6: the 5-comparator network sorts 4 wires in depth 3")
assert all(any(c == (i, i + 1) for L in bitonic(8) for c in L) for i in range(7))
print("Exercise 2.7: the bitonic network on 8 wires contains every adjacent comparator")

def insertion_net(n):
    comps = []
    for k in range(1, n):
        comps += [(j, j + 1) for j in range(k - 1, -1, -1)]
    return [[c] for c in comps]
for n in range(2, 11):
    net_ = insertion_net(n); assert sorts01(net_, n) and depth(net_, n) == 2 * n - 3
print("Exercise 2.8: insertion networks sort, with depth 2n-3, for n = 2..10")

### P2.6 Horses

In [ ]:
def seven_races(speed):
    race = lambda hs: sorted(hs, key=lambda h: -speed[h])
    groups = [race(range(5 * g, 5 * g + 5)) for g in range(5)]
    order = race([g[0] for g in groups])
    G = {g[0]: g for g in groups}
    A = order; top = A[0]
    cands = [A[1], A[2], G[A[0]][1], G[A[0]][2], G[A[1]][1]]
    second, third = race(cands)[:2]
    return [top, second, third]

for _ in range(20000):
    speed = rng.permutation(25)
    assert seven_races(speed) == sorted(range(25), key=lambda h: -speed[h])[:3]
print("the seven-race method found the three fastest in 20000 random races")